In [16]:
import pandas as pd
import statsmodels.api as sm

In [8]:
df = pd.read_excel('data/life_expectancy.xls')

In [9]:
df

,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Measles,BMI,under-five deaths,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,65.0,201.50,62,2.50,71.279624,1154,76.50,83,6,8.16,93.50,0.1,584.259210,33736494,17.2,17.3,0.9800,10.1
1,59.9,195.89,64,9.13,73.523582,492,86.19,86,58,8.18,86.87,0.1,612.696514,327582,17.5,17.5,0.9188,10.0
2,59.9,195.89,66,9.13,73.219243,430,60.96,89,62,8.13,86.87,0.1,631.744976,31731688,17.7,17.7,0.9188,9.9
3,59.5,195.45,69,9.65,78.184215,2787,60.80,93,67,8.52,86.35,0.1,669.959000,3696958,17.9,18.0,0.9140,9.8
4,59.2,195.12,71,10.04,7.097109,3013,60.68,97,68,7.87,85.96,0.1,63.537231,2978599,18.2,18.2,0.9104,9.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,76.2,213.82,8,16.18,316.680000,1,48.56,9,92,4.79,109.68,0.1,12245.256450,42981515,1.0,0.9,1.6478,17.3
82,76.0,213.60,8,16.40,1001.796332,0,48.80,10,99,4.99,109.40,0.1,12976.636420,42539925,1.0,0.9,1.6440,17.3
83,75.9,213.49,9,16.51,1133.558003,2,48.92,10,99,5.20,109.26,0.1,12969.771200,4296739,1.0,0.9,1.6421,17.2
84,75.7,188.84,9,16.73,1504.329462,3,49.16,10,93,5.89,108.98,0.1,12726.983600,41656879,1.0,0.9,1.6383,17.1


а). Построим линейную регрессию, выявим выжные коэффициенты при альфа = 0.2

In [12]:
y = df['Life expectancy ']
X = df.drop(columns=['Life expectancy '])

In [17]:
X = sm.add_constant(X)

In [18]:
model = sm.OLS(y, X).fit()

In [19]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       Life expectancy    R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.806e+04
Date:                Sat, 25 Oct 2025   Prob (F-statistic):          2.86e-117
Time:                        20:54:32   Log-Likelihood:                 35.320
No. Observations:                  86   AIC:                            -34.64
Df Residuals:                      68   BIC:                             9.537
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const     

In [20]:
alpha = 0.2
z = model.pvalues[model.pvalues <= alpha].index.tolist()
print(z)

['const', ' BMI ', 'Diphtheria ', 'Income composition of resources', 'Schooling']


б). тест Бреуша–Пагана

In [22]:
from scipy.stats import chi2
import numpy as np

In [ ]:
residuals = model.resid

e2 = residuals ** 2
aux_model = sm.OLS(e2, model.model.exog).fit()
R2_aux = aux_model.rsquared

n = len(e2)
k = model.model.exog.shape[1] - 1
LM = n * R2_aux
p_value = 1 - chi2.cdf(LM, k)

print(LM)
print(k)
print(p_value.round(4))

54.649024552304034
17
0.0


p_value < alpha, значит гетероскедастичность присутствует (отклоняем H0)

в). тест Уайта

In [31]:
from itertools import combinations

residuals = model.resid
X = model.model.exog  
n, k_plus1 = X.shape
k = k_plus1 - 1

X_no_const = X[:, 1:]
X_white = [np.ones((n, 1))]  
X_white.append(X_no_const)
X_white.append(X_no_const)

X_white.append(X_no_const ** 2)

for i, j in combinations(range(X_no_const.shape[1]), 2):
    X_white.append((X_no_const[:, i] * X_no_const[:, j]).reshape(-1, 1))

X_white = np.hstack(X_white)

e2 = residuals ** 2
aux_model = sm.OLS(e2, X_white).fit()
R2_aux = aux_model.rsquared

LM = n * R2_aux
df = X_white.shape[1] - 1
p_value = 1 - chi2.cdf(LM, df)

print(LM)
print(k)
print(p_value.round(4))


85.99108536095626
17
1.0


p_value > alpha, значит нет гетероскедастичности

г). тест Голдфелда–Квандта

In [33]:
from scipy.stats import f

In [35]:
df = pd.read_excel('data/life_expectancy.xls')
df.columns = df.columns.str.strip().str.lower()

In [41]:
sort_var = 'gdp'
df_sorted = df.sort_values(by=sort_var).reset_index(drop=True)

y_sorted = df_sorted['life expectancy']
X_sorted = df_sorted.drop(columns=['life expectancy'])
X_sorted = sm.add_constant(X_sorted)

drop_ratio = 0.2
n = len(df_sorted)
drop_n = int(n * drop_ratio / 2)

low_idx = slice(0, n//2 - drop_n)
high_idx = slice(n//2 + drop_n, n)

X1, y1 = X_sorted.iloc[low_idx, :], y_sorted.iloc[low_idx]
X2, y2 = X_sorted.iloc[high_idx, :], y_sorted.iloc[high_idx]

model1 = sm.OLS(y1, X1).fit()
model2 = sm.OLS(y2, X2).fit()

SSR1 = np.sum(model1.resid ** 2)
SSR2 = np.sum(model2.resid ** 2)
df1 = len(y1) - X1.shape[1]
df2 = len(y2) - X2.shape[1]

F_stat = (SSR2 / df2) / (SSR1 / df1)
p_value = 1 - f.cdf(F_stat, df2, df1)

print(F_stat)
print(p_value)


6.816698267601534
0.00012948248129251994


p_value < alpha, значит гетероскедастичность присутствует (отклоняем H0)